# 06 — Deploy artifacts + offline live-replay benchmark

- Copies `deploy_template/{helium_emotion.py, integrate_helium_emotion.py}` to
  `deploy_out/` and verifies the deploy head class loads the final checkpoint.
- CPU smoke test of `HeliumEmotionManager` (the `AutoEmotionManager`-compatible
  drop-in: `start/feed/feed_step/current_emotion_id/reset_session/stop`).
- Offline replay on in-domain test features: per-step latency, VRAM, switches/min.
- Prints the copy-to-live checklist. Optional HF upload (flag).


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
UPLOAD_HF = False
HF_UPLOAD_REPO = "YOUR_USER/helium-emotion-head"   # change me
TEMPLATE_DIR = Path("deploy_template")             # ships with these notebooks
final = P.MODELS / "helium_emotion_head_final.pt"
assert final.exists(), "run 05 first"
assert TEMPLATE_DIR.exists(), "deploy_template/ must sit next to this notebook"


In [ ]:
import shutil, ast
P.DEPLOY.mkdir(parents=True, exist_ok=True)
for f in ("helium_emotion.py", "integrate_helium_emotion.py"):
    shutil.copy2(TEMPLATE_DIR / f, P.DEPLOY / f)
    ast.parse((P.DEPLOY / f).read_text(encoding="utf-8"))
    print("deploy file OK:", P.DEPLOY / f)

sys.path.insert(0, str(P.DEPLOY))
import importlib
import helium_emotion as he_deploy
importlib.reload(he_deploy)
payload = torch.load(final, map_location="cpu", weights_only=False)
hd = he_deploy.EmotionFusionHead(**payload["cfg"])
hd.load_state_dict(payload["state_dict"])   # raises on any mismatch
print("deploy head loads the final checkpoint: OK")


In [ ]:
# CPU smoke test of the AutoEmotionManager-compatible drop-in
class _A:
    pass
a = _A()
a.emotion_id = 0
a.helium_emotion = True
a.helium_head_path = str(final)
a.helium_head_device = "cpu"
a.helium_temperature = 1.0
a.helium_ema_alpha = 0.4
a.helium_enter_margin = 0.15
a.helium_min_dwell_s = 1.5
mgr = he_deploy.HeliumEmotionManager(a)
mgr.start()
mgr.feed(b"\x00" * 1920, 24000)          # mic shim: must be a no-op, never raise
pcm = (np.random.randn(1920) * 0.05).astype(np.float32)
for t in range(25):
    mgr.feed_step(torch.randn(1, eh.HELIUM_DIM), None, pcm)
print("manager smoke:", mgr.current_emotion_id(), mgr.current_emotion(),
      "| steps:", mgr.steps_fed, "| errors:", mgr.feed_errors)
mgr.reset_session()
print("after reset:", mgr.current_emotion_id(), mgr.current_emotion())
mgr.stop()


In [ ]:
# offline replay on in-domain features (stored pros/latents — full fidelity)
recs = [r for r in eh.iter_shard_records(P.FEAT_B, "phaseB")
        if r["split"] == "test_indomain"][:24]
if not recs:
    recs = list(eh.iter_shard_records(P.FEAT_B, "phaseB"))[:24]
assert recs, "no phaseB features — run 05 first"
runner = he_deploy.EmotionHeadRunner(final, device=DEVICE)
cfg = json.loads((P.MODELS / "config.json").read_text())["smoother"]
sm = he_deploy.HysteresisSmoother(ema_alpha=cfg["ema_alpha"],
                                  enter_margin=cfg["enter_margin"],
                                  min_dwell_s=cfg["min_dwell_s"])
print("smoother:", cfg)
if DEVICE == "cuda":
    h = torch.randn(1, eh.HELIUM_DIM, device=DEVICE)
    m = torch.randn(1, eh.MIMI_LATENT_DIM, device=DEVICE)
    p = torch.randn(1, eh.PROSODY_DIM, device=DEVICE)
    hh = None
    for _ in range(20):
        _, hh = runner.head.forward_step(h, m, p, hh)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    N = 1000
    for _ in range(N):
        _, hh = runner.head.forward_step(h, m, p, hh)
    torch.cuda.synchronize()
    print(f"head forward_step: {(time.perf_counter()-t0)/N*1e6:.1f} µs/step | "
          f"vram {torch.cuda.memory_allocated()/1e6:.0f} MB")
sw = []
for r in recs:
    sm.reset()
    runner.reset()
    T = r["hidden"].shape[0]
    hh = None
    for t in range(T):
        if not bool(r["agent_mask"][t]):
            continue
        h = r["hidden"][t].float().unsqueeze(0).to(DEVICE)
        m = r["mimi_lat"][t].float().unsqueeze(0).to(DEVICE)
        p = r["pros"][t].float().unsqueeze(0).to(DEVICE)
        logits, hh = runner.head.forward_step(h, m, p, hh)
        sm.update(torch.softmax(logits.float(), -1)[0].cpu().numpy())
    sw.append(sm.switches_per_min())
print(f"replay on {len(sw)} in-domain clips: switches/min "
      f"mean={np.mean(sw):.2f} p95={np.percentile(sw, 95):.2f}")


In [ ]:
if UPLOAD_HF:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_UPLOAD_REPO, exist_ok=True)
    for f in ("helium_emotion_head_final.pt", "config.json"):
        api.upload_file(path_or_fileobj=str(P.MODELS / f), path_in_repo=f,
                        repo_id=HF_UPLOAD_REPO)
    print("uploaded to", HF_UPLOAD_REPO)
print("""
DEPLOY CHECKLIST (on the live machine):
1. copy deploy_out/helium_emotion.py + deploy_out/integrate_helium_emotion.py
   next to the server file (IMTalker/)
2. copy models/helium_emotion_head_final.pt (+ config.json) there too
3. python integrate_helium_emotion.py <server_file.py>   # idempotent; writes .bak_helium
4. start the server with:
   --helium_emotion --helium_head_path helium_emotion_head_final.pt
5. watch for [HELIUM-EMOTION] switch logs; tune --helium_ema_alpha /
   --helium_enter_margin / --helium_min_dwell_s from config.json values
""")
